# Welcome to the `go` model linear programming tutorial!

## A brief introduction

We will be running a linear programming model for the Western Electricity Coordinating Council (WECC) region.

## Install `go` from GitHub

```bash
python -m pip install -e git://github.com/IMMM-SFA/go.git@main#egg=go

```

## Load packages

In [ ]:
import pyomo.environ as pyo

from go.wecc import LinearProgrammingModel
# from go.wecc import MixedIntegerModel


## Illustrative subset of data

In [ ]:
# test WECC data
wecc_data = '/Users/d3y010/repos/github/IM3_WECC/Model/WECC_data.dat'


## Create .dat file

This file is created by another code that combines inputs of different CSV files.

From this script:  `/Users/d3y010/repos/github/IM3_WECC/Model/WECCDataSetup.py`

Explore this one to see contents `/Users/d3y010/repos/github/IM3_WECC/Model/UC.dat`



## Run linear programming model

In [ ]:
model = LinearProgrammingModel(data=wecc_data)

model


In [ ]:
inst = model.create_instance()


## Setup the solver

In [ ]:
opt = pyo.SolverFactory('glpk')


## Need to download `cbc` executable

Look into threading:  `opt.solve(instance, options={"threads": 4})`


In [ ]:
opt.solve(inst, tee=True)


In [ ]:
inst.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)


In [ ]:
days = 4

H = inst.HorizonHours
D = 2
K=range(1,H+1)


In [ ]:
for day in range(1, days):
    
    for z in inst.busses:
        
        # load demand and reserve time series data
        for i in K:
            
            inst.HorizonDemand[z, i] = instance.SimDemand[z, (day-1) * 24 + i]
            
            
    for z in inst.Hydro:
        
        # load hydropower time series data
        inst.HorizonHydro[z] = instance.SimHydro[z, day]
        

    for z in inst.Solar:
        
        # load solar time series data
        inst.HorizonSolar[z, i] = instance.SimSolar[z, (day-1) * 24 + i]
        
    for z in inst.Wind:
        
        # load wind time series data
        inst.HorizonWind[z, i] = instance.SimWind[z, (day-1) * 24 + i]
        
    result = opt.solve(instance,tee=True,symbolic_solver_labels=True) ##,tee=True to check number of variables\n",
    instance.solutions.load_from(result) 
        
        
        
        
        